# Plantilla de Proof of Concept — versión simplificada completada
### Sesión 4 · De las fichas al código · Stack LangChain

Este notebook es el puente entre el diseño y la implementación. En esta versión se mantiene el desarrollo lo más simple posible: usamos un CSV ficticio, SQLite y un agente mínimo que razona sobre datos estructurados.

1. Ficha 1: define el caso de uso, la política de incertidumbre y los criterios de calidad.
2. Ficha 2: prioriza la feature y marca la arquitectura por capas.
3. Ficha 3: define la fuente de datos, el tratamiento ETL y la trazabilidad.

La idea es validar el flujo antes de pensar en una versión más sofisticada.

### El mapa: cuatro capas, de abajo hacia arriba

El notebook sigue el mismo orden de la arquitectura Medallion que ya conoces: el dato fluye limpio hacia arriba y el agente solo consume lo que la capa de datos le expone.

Primero preparas el **entorno** y eliges tu modelo. Luego construyes la **capa de datos**, donde cada fuente de tu Ficha 3 se convierte en una *tool* que el agente puede llamar (RAG, SQL, APIs externas). Sobre ella montas la **capa de agente**, que es el cerebro: el *prompt* que define su rol y reglas, más la orquestación que decide qué tool usar. Y al final lo pruebas todo en un **playground** con un pequeño set de casos derivado de tu Ficha 1.

---

## 0 · Configuración del entorno

Instalamos la base común. **LangChain** aporta las abstracciones de modelo, *prompt* y agente; **langchain-community** trae las integraciones (cargadores de datos, *vector stores*, utilidades SQL); y **langsmith** te da observabilidad: registra cada paso del agente para que puedas depurarlo.

En 2026, el agente de `create_agent` corre sobre **LangGraph** por debajo, así que LangChain ya lo trae como dependencia. Más abajo dejamos comentados los *extras* que activarás según tu caso: proveedor del modelo, *vector store*, driver de base de datos y tools externas.

In [1]:
# 0.1 · Dependencias base del template
# Este PoC usa LangChain, Gemini y LangSmith como stack real.
# Si ejecutas el notebook desde cero, instala los paquetes necesarios antes de correr las celdas de agente.
# %pip install -qU langchain langchain-community langchain-google-genai langsmith python-dotenv


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### 0.2 · Claves y trazabilidad

Carga las claves de forma segura (sin escribirlas en el notebook) y enciende **LangSmith**. Con la traza activa, cada ejecución de las siguientes celdas quedará registrada en tu proyecto: qué tools llamó el agente, con qué argumentos, cuánto tardó y dónde falló. Es la observabilidad de tu Ficha 3, ahora aplicada al agente.

In [4]:
# 0.2 · Configuración y rutas
import json
import os
import re
import sqlite3
import time
import uuid
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

BASE_DIR = Path.cwd()
if (BASE_DIR / 'gastos_ficticios_poc.csv').exists():
    DATA_DIR = BASE_DIR
elif (BASE_DIR / 'Recursos y Contexto' / 'gastos_ficticios_poc.csv').exists():
    DATA_DIR = BASE_DIR / 'Recursos y Contexto'
elif (BASE_DIR.parent / 'Recursos y Contexto' / 'gastos_ficticios_poc.csv').exists():
    DATA_DIR = BASE_DIR.parent / 'Recursos y Contexto'
else:
    DATA_DIR = BASE_DIR / 'Recursos y Contexto'
CSV_PATH = DATA_DIR / 'gastos_ficticios_poc.csv'
DB_PATH = DATA_DIR / 'poc_gastos.sqlite'

load_dotenv(BASE_DIR / '.env')
load_dotenv(BASE_DIR.parent / '.env')

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY')
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')

if not GOOGLE_API_KEY:
    raise RuntimeError('Falta GOOGLE_API_KEY o GEMINI_API_KEY en .env')
if not LANGSMITH_API_KEY:
    raise RuntimeError('Falta LANGSMITH_API_KEY en .env')

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY
os.environ.setdefault('LANGSMITH_TRACING', 'true')
os.environ.setdefault('LANGSMITH_PROJECT', 'PoC-ProyectoFinal-BSG')
os.environ.setdefault('LANGCHAIN_PROJECT', 'PoC-ProyectoFinal-BSG')

MARGEN_UTILIDAD = 0.35
PROJECT_ID_RE = re.compile(r'\b\d{4}-\d{4}\b')

print('Variables cargadas desde .env y trazas activadas en LangSmith.')


### 0.3 · Tu modelo de lenguaje

El modelo es el motor de razonamiento del agente. Elige el de tu Ficha 2. `init_chat_model` acepta un identificador con el formato `"<proveedor>:<modelo>"` y te devuelve un objeto de chat homogéneo, así puedes cambiar de proveedor tocando una sola línea.

In [3]:
# 0.3 · Modelo
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL_ID = 'gemini-2.5-flash'
llm = ChatGoogleGenerativeAI(
    model=MODEL_ID,
    temperature=0,
    thinking_budget=0,
    max_retries=2,
    metadata={'ls_model_name': MODEL_ID},
)
print(f'Modelo seleccionado: {MODEL_ID}')


/Users/isma/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/isma/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/isma/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then

OK


### 0.4 · Declara el stack de tu caso

Antes de codear, dejamos por escrito qué piezas usa este PoC.

- **Modelo:** agente mínimo basado en reglas sobre datos estructurados.
- **RAG (embeddings + vector store):** no aplica.
- **Base de datos (SQL):** SQLite local para pruebas.
- **Tools externas / APIs:** no aplica en esta versión.
- **Política de incertidumbre:** si falta el proyecto o la cantidad es inconsistente, el campo queda en `null` y se reporta en `campos_baja_confianza`.


## 1 · Capa de Datos: procesamiento y preparación de *tools*

Esta capa convierte tu Ficha 3 en código. La idea central: **cada fuente de datos se transforma en una capacidad que el agente puede invocar**, es decir, una *tool*. Cubrimos las tres formas más comunes —recuperación sobre texto (RAG), consulta estructurada (SQL) y herramientas externas— pero activa solo las que tu caso necesite.

Recuerda la regla de la arquitectura Medallion: el agente consume lo que esta capa **expone limpio**, nunca toca la fuente cruda directamente.

### 1.1 · RAG — recuperación sobre documentos

Convierte los documentos ya limpios de tu capa *Curated* (Ficha 3) en una *tool* de búsqueda semántica. La descripción de la tool es clave: el agente decide cuándo usarla leyéndola, así que sé específico sobre **qué contiene** y **cuándo conviene**.

In [ ]:
# 1.1 · RAG
# No aplica en este PoC porque la fuente principal es estructurada y vive en SQLite.
rag_tool = None


### 1.2 · SQL — consulta a datos estructurados

Da al agente acceso de **solo lectura** a tu base estructurada. Puedes envolver la consulta en una *tool* propia (control fino) o usar el *toolkit* de SQL, que entrega varias tools listas para explorar el esquema y consultar.

In [ ]:
# 1.2 · SQL

def abrir_db():
    return sqlite3.connect(DB_PATH)


def bootstrap_db():
    df = pd.read_csv(CSV_PATH)
    with abrir_db() as conn:
        df.to_sql('gastos_proyecto', conn, if_exists='replace', index=False)
        conn.execute('DROP VIEW IF EXISTS view_gastos_proyecto')
        conn.execute(
            '''
            CREATE VIEW view_gastos_proyecto AS
            SELECT
                idproyecto,
                claveproducto,
                nombreproducto,
                fechaproyecto,
                COUNT(*) AS cantidad_gastos,
                SUM(monto_gasto) AS total_gastos
            FROM gastos_proyecto
            GROUP BY idproyecto, claveproducto, nombreproducto, fechaproyecto
            '''
        )
        conn.execute(
            '''
            CREATE TABLE IF NOT EXISTS logs_inferencia (
                run_id TEXT PRIMARY KEY,
                timestamp_ejecucion TEXT,
                idproyecto TEXT,
                entrada_texto TEXT,
                salida_json TEXT,
                modelo TEXT,
                latencia_ms REAL,
                score_global REAL,
                estado TEXT
            )
            '''
        )


def consultar_sql(query: str) -> str:
    with abrir_db() as conn:
        resultado = pd.read_sql_query(query, conn)
    return resultado.to_json(orient='records', force_ascii=False)


def obtener_gastos_proyecto(idproyecto: str) -> pd.DataFrame:
    with abrir_db() as conn:
        return pd.read_sql_query(
            'SELECT * FROM gastos_proyecto WHERE idproyecto = ?',
            conn,
            params=(idproyecto,),
        )


def registrar_log(idproyecto: str, entrada: str, salida: dict, latencia_ms: float, score_global: float, estado: str = 'READY') -> None:
    run_id = str(uuid.uuid4())
    payload = (
        run_id,
        pd.Timestamp.now('UTC').isoformat(),
        idproyecto,
        entrada,
        json.dumps(salida, ensure_ascii=False),
        globals().get('MODEL_ID', 'poc-reglas-sqlite'),
        latencia_ms,
        score_global,
        estado,
    )
    with abrir_db() as conn:
        conn.execute(
            '''
            INSERT OR REPLACE INTO logs_inferencia
            (run_id, timestamp_ejecucion, idproyecto, entrada_texto, salida_json, modelo, latencia_ms, score_global, estado)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''',
            payload,
        )
        conn.commit()

bootstrap_db()
print('SQLite lista en:', DB_PATH)


In [ ]:
# 1.2.1 · Lógica de cálculo del PoC

def formatear_respuesta_natural(salida: dict) -> str:
    if not salida.get('idproyecto'):
        return (
            'No pude recuperar un proyecto valido con la informacion disponible. '
            'Por ahora no encuentro data suficiente para describir gastos, costo o precio con confianza.\n\n'
            'En este caso revise la consulta sobre la base de datos, pero no aparecio un identificador de proyecto que permitiera completar la busqueda.\n\n'
            'Si me compartes un idproyecto valido, puedo volver a consultar la base y devolverte una estimacion clara y trazable.'
        )

    linea_datos = (
        f"Recupere los datos del proyecto {salida['idproyecto']}, asociado a la clave {salida['claveproducto']} y al material {salida['nombreproducto']}. "
        f"Para este proyecto identifique {salida['cantidad_gastos']} gastos registrados con un total acumulado de {salida['total_gastos']:.2f}."
    )
    linea_costos = (
        f"Con esa informacion estime un costo propuesto de {salida['propuesta_costo']:.2f} y un precio sugerido de {salida['propuesta_precio']:.2f}, "
        f"usando la regla de margen definida para el PoC. El nivel de confianza global quedo en {salida['score_global']:.2f}."
    )
    if salida.get('campos_baja_confianza'):
        linea_origen = (
            'La informacion se obtuvo al consultar la base SQLite del proyecto y luego consolidar los gastos por proyecto. '
            f"Detecte baja confianza en {', '.join(salida['campos_baja_confianza'])}, por lo que esos campos se reportan con cautela."
        )
    else:
        linea_origen = (
            'La informacion se obtuvo al consultar directamente la base SQLite cargada desde el CSV de prueba, '
            'y despues aplicar la logica de calculo del PoC sin inventar valores adicionales.'
        )

    return f"{linea_datos}\n\n{linea_costos}\n\n{linea_origen}"


def calcular_resultado(idproyecto: str, entrada_texto: str) -> dict:
    inicio = time.perf_counter()
    df = obtener_gastos_proyecto(idproyecto)

    if df.empty:
        salida = {
            'idproyecto': idproyecto,
            'claveproducto': None,
            'nombreproducto': None,
            'propuesta_costo': None,
            'propuesta_precio': None,
            'cantidad_gastos': 0,
            'total_gastos': 0.0,
            'score_global': 0.0,
            'campos_baja_confianza': ['propuesta_costo', 'propuesta_precio'],
        }
        registrar_log(idproyecto, entrada_texto, salida, (time.perf_counter() - inicio) * 1000, 0.0, 'NO_DATA')
        return salida

    claveproducto = str(df['claveproducto'].iloc[0]).strip()
    nombreproducto = str(df['nombreproducto'].iloc[0]).strip()
    cantidad_gastos = int(df.shape[0])
    total_gastos = float(df['monto_gasto'].sum())

    cantidades_validas = pd.to_numeric(df['cantidad'], errors='coerce').dropna().astype(int)
    campos_baja_confianza = []
    if cantidades_validas.empty:
        cantidad_base = None
        campos_baja_confianza.append('cantidad')
    else:
        cantidad_base = int(cantidades_validas.iloc[0])
        if cantidades_validas.nunique() > 1:
            campos_baja_confianza.append('cantidad')

    if cantidad_base and cantidad_base > 0:
        propuesta_costo = round(total_gastos / cantidad_base, 2)
        propuesta_precio = round(propuesta_costo * (1 + MARGEN_UTILIDAD), 2)
    else:
        propuesta_costo = None
        propuesta_precio = None
        campos_baja_confianza.extend(['propuesta_costo', 'propuesta_precio'])

    score_global = 0.95
    if 'cantidad' in campos_baja_confianza:
        score_global -= 0.15
    if propuesta_costo is None:
        score_global = 0.0
    score_global = max(0.0, round(score_global, 2))

    salida = {
        'idproyecto': idproyecto,
        'claveproducto': claveproducto,
        'nombreproducto': nombreproducto,
        'propuesta_costo': propuesta_costo,
        'propuesta_precio': propuesta_precio,
        'cantidad_gastos': cantidad_gastos,
        'total_gastos': round(total_gastos, 2),
        'score_global': score_global,
        'campos_baja_confianza': sorted(set(campos_baja_confianza)),
    }
    registrar_log(idproyecto, entrada_texto, salida, (time.perf_counter() - inicio) * 1000, score_global)
    return salida


### 1.3 · Tools externas o propias

Cualquier otra capacidad que tu arquitectura (Ficha 2) defina: llamar a una API, hacer un cálculo, ejecutar una acción sobre otro sistema. Se declara con el decorador `@tool`. El *docstring* no es decorativo: es lo que el agente lee para decidir si la usa.

In [ ]:
# 1.3 · Tool externa / propia

def resumen_proyecto_sql(idproyecto: str) -> str:
    """Recupera y consolida los gastos de un proyecto editorial desde SQLite."""
    resumen = calcular_resultado(idproyecto, f'Consulta del proyecto {idproyecto}')
    return json.dumps(resumen, ensure_ascii=False, indent=2)


### 1.4 · Reúne tus tools

Junta en una sola lista las tools que **sí** activaste arriba. Esta lista es lo único que la capa de agente necesita de aquí.

In [ ]:
# 1.4 · Lista de tools activas
tools = [
    resumen_proyecto_sql,
]
print(f'{len(tools)} tool(s) lista(s):', [getattr(t, 'name', getattr(t, '__name__', str(t))) for t in tools])


---

## 2 · Capa de Agente: prompt y orquestación

Ahora el cerebro. El *system prompt* traduce tu Ficha 1 a instrucciones: quién es el agente, cuál es su tarea y qué reglas respeta (sobre todo tu política de *"nunca inventar"*). La orquestación —el bucle *razonar → actuar → observar*— la arma `create_agent`, la API vigente de LangChain en 2026, que corre sobre LangGraph por debajo.

In [ ]:
# 2.1 · System prompt
SYSTEM_PROMPT = """
Eres un asistente editorial especializado en costos y precios de venta.
Tu objetivo es ayudar a calcular el costo y sugerir un precio a partir de los gastos registrados en la base.

Reglas:
- Usa la herramienta `resumen_proyecto_sql` cuando el usuario proporcione un idproyecto.
- No inventes datos ni supongas valores que no estén en la base.
- Responde siempre en lenguaje natural, con exactamente 3 parrafos y sin listas ni formato muy estructurado.
- El primer parrafo debe explicar la data recuperada.
- El segundo parrafo debe explicar el costo y el precio estimados.
- El tercer parrafo debe explicar brevemente como obtuviste la informacion y mencionar la confianza o las dudas, si las hay.
- Si no encuentras un proyecto valido, pide el dato faltante de forma natural.
""".strip()

print(SYSTEM_PROMPT)


In [ ]:
# 2.2 · Construccion del agente
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    name='poc_costos_editoriales',
)
print('Agente LangChain listo con Gemini 2.5 Flash y LangSmith.')


### ¿Cuándo subir de `create_agent` a LangGraph?

`create_agent` te da el bucle *razonar → actuar → observar* listo para usar y alcanza para la mayoría de los PoC. El momento de bajar a un `StateGraph` explícito de LangGraph llega cuando tu caso necesita ramas condicionales, reintentos basados en resultados intermedios, una pausa para aprobación humana antes de una acción sensible, o memoria persistente entre sesiones. No empieces ahí: súbete de nivel cuando el PoC te lo pida.

---

## 3 · Test y Playground

Probamos el agente de tres formas: una invocación simple, un pequeño set de casos derivado de tu Ficha 1 (la meta no es *"que suene bien"*, sino verificar el comportamiento esperado), y un playground interactivo para explorar.

In [ ]:
# 3.1 · Invocacion simple
def preguntar(texto: str) -> str:
    salida = agent.invoke({'messages': [{'role': 'user', 'content': texto}]})
    if hasattr(salida, 'content'):
        return salida.content
    if isinstance(salida, dict):
        mensaje_final = salida['messages'][-1]
        if hasattr(mensaje_final, 'content'):
            return mensaje_final.content
        return mensaje_final['content']
    return str(salida)

print(preguntar('Calcula el costo y precio del proyecto 3232-0001'))


In [ ]:
# 3.2 · Mini set de prueba
casos = [
    'Calcula el costo y precio del proyecto 3232-0002',
    'Calcula el costo y precio del proyecto 3232-0004',
    'Calcula el costo y precio sin id de proyecto',
]
for i, c in enumerate(casos, 1):
    print(f'\n=== Caso {i} ===\n{c}\n---')
    print(preguntar(c))


In [ ]:
# 3.3 · Playground interactivo
while True:
    q = input('\nTu > ')
    if q.strip().lower() in {'salir', 'exit', 'quit'}:
        break
    print('Agente >', preguntar(q))


### 3.4 · Observabilidad

Si activaste LangSmith en el paso 0.2, todas las ejecuciones de arriba quedaron registradas en tu proyecto en [smith.langchain.com](https://smith.langchain.com). Ahí puedes ver el árbol de cada corrida: qué tools llamó el agente, con qué argumentos, la latencia por paso y dónde falló. Úsalo para depurar el comportamiento, no solo el resultado final.

---

## 4 · Próximos pasos

Este PoC valida tu tesis de principio a fin con, al menos, un camino feliz. Desde aquí, el trabajo se endurece: robustece la capa de datos con lo que diseñaste en la Ficha 3 (manejo de errores con *DLQ*, idempotencia, trazabilidad), amplía el set de pruebas más allá del caso feliz, y —solo si el flujo de control lo exige— migra el agente a un `StateGraph` de LangGraph.

Tu PoC está listo para convertirse en MVP cuando puedes afirmar que:

1. El agente resuelve al menos un caso real de principio a fin usando tus tools.
2. Reconoce cuándo no sabe y respeta tu política de *"nunca inventar"*.
3. Tienes trazas en LangSmith para depurar su comportamiento.
4. Sabes qué pieza endurecer primero para pasar de PoC a producción.

> Las tres fichas definieron el *qué* y el *con qué*. Este notebook arranca el *cómo*. A partir de aquí, construir es ejecución informada, no improvisación.